In [ ]:
from sklearn.model_selection import cross_val_score, KFold, cross_validate
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.feature_selection import VarianceThreshold, RFECV, RFE
from sklearn.linear_model import LinearRegression


from ml_enhance import plot_scaled_linreg_result, CorrelationFilter


import matplotlib.pyplot as plt
from sklearn import pipeline
import seaborn as sns
import pandas as pd
import numpy as np
import scipy
import json

In [ ]:
def make_pipeline(model):
    return pipeline.Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("remove_corr", CorrelationFilter(threshold=1.0)),
        ("transform", PowerTransformer(method='yeo-johnson', standardize=False)),
        ("scale", RobustScaler()),
        ("predict", model)
    ])

In [94]:
df = pd.read_csv("../data/processed_dataset_wo_metals.csv")

df = df.drop("avg_atomic_quadrupole_principal_invariant_3", axis=1)

y = df["solubility"]
X = df.drop(["solubility", "smiles", "canon_smiles", "id"], axis=1)

In [95]:
with open("../data/rdkit_feature_names.json", "r") as f:
    rdkit_feature_list: list = json.load(f)

mask = X.columns.isin(rdkit_feature_list)

In [96]:
X_topo = X.iloc[:, mask]
X_qm = X.iloc[:, ~mask]

In [97]:
pl_combo = make_pipeline(LinearRegression())
pl_topo = make_pipeline(LinearRegression())
pl_qm = make_pipeline(LinearRegression())

In [98]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [99]:
scoring = {
    "r2": "r2",
    "MSE": "neg_mean_squared_error"
}

In [100]:
scores_combo = cross_validate(pl_combo, X, y, cv=cv, scoring=scoring, n_jobs=4, return_estimator=True)

In [101]:
scores_topo = cross_validate(pl_topo, X_topo, y, cv=cv, scoring=scoring, n_jobs=4, return_estimator=True)

In [102]:
scores_qm = cross_validate(pl_qm, X_qm, y, cv=cv, scoring=scoring, n_jobs=4, return_estimator=True)

In [103]:
print(f"Topology alone: {scores_topo["test_r2"].mean()}\nQM alone: {scores_qm["test_r2"].mean()}\nCombined: {scores_combo["test_r2"].mean()}")

Topology alone: 0.8198113594610325
QM alone: 0.7363322024985696
Combined: 0.8210639921411286


In [104]:
best_pl = scores_combo["estimator"][1]

best_pl[:-1].transform(X).shape

(8763, 267)

For the linear regression, the QM descriptors alone seem to give the worst performance out of the three, but **there seems to be no significant difference between the topological descriptors alone and the combined feature set**. This can mean three things, the QM descriptors provide no advantage because:
- due to the larger dataset (desired outcome)
- They are not so useful for solubility predictions
- The linear model cannot benefit from using them (in one way or another) 

Lets try and make a RFE to see whether the QM descriptors do in fact provide any value to the linear model.

In [105]:
def make_rfe_pipeline(model, cv=cv):
    return pipeline.Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("remove_corr", RemoveCorrelatedFeatures(threshold=0.99)),
        ("transform", PowerTransformer(method='yeo-johnson', standardize=False)),
        ("scale", RobustScaler()),
        ("rfe", RFECV(model, cv=cv, scoring="r2", n_jobs=4))
    ])

In [106]:
rfe_pipe = make_rfe_pipeline(LinearRegression())
rfe_pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('variance', ...), ('remove_corr', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",0.0
,threshold,0.99
,"method method: {'yeo-johnson', 'box-cox'}, default='yeo-johnson'The power transform method. Available methods are:- 'yeo-johnson' [1]_, works with positive and negative values- 'box-cox' [2]_, only works with strictly positive values",'yeo-johnson'
,"standardize standardize: bool, default=TrueSet to True to apply zero-mean, unit-variance normalization to thetransformed output.",False
,"copy copy: bool, default=TrueSet to False to perform inplace computation during transformation.",True
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True


In [107]:
rfe_pipe.get_feature_names_out()

array(['energy', 'atomization_energy', 'homo_lumo_gap',
       'ionization_energy', 'chemical_potential', 'molecular_dipole_norm',
       'molecular_polarizability_mean', 'enthalpy', 'zero_point_energy',
       'radius_of_gyration', 'molecular_volume', 'sterimol_L',
       'ir_mode_count_1500', 'ir_centroid_freq_1500',
       'ir_mode_count_1500_2750', 'ir_centroid_freq_1500_2750',
       'ir_centroid_freq_2750_4000', 'avg_partial_charge_cyclohexane',
       'avg_effective_coordination_number',
       'avg_atomic_polarizability_anisotropy',
       'avg_percentage_buried_volume', 'avg_atomic_sasa',
       'avg_atomic_dipole_norm', 'avg_atomic_fukui_minus',
       'avg_atomic_quadrupole_principal_invariant_2',
       'avg_atomic_polarizability_mean', 'avg_bond_stiffness',
       'avg_bond_length', 'avg_nuclear_repulsion', 'avg_overlap_integral',
       'avg_bond_energy', 'qed', 'MolWt', 'NumRadicalElectrons',
       'MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge',
       'M

In [108]:
rfe = rfe_pipe.named_steps['rfe']

importance = pd.Series(
    rfe.estimator_.coef_,
    index=rfe_pipe.get_feature_names_out()
)
importance.sort_values(ascending=False)

fr_phos_ester        154.403392
fr_benzodiazepine     84.515300
fr_quatN              69.213253
fr_SH                 43.411669
fr_amidine            43.399279
                        ...    
fr_hdrzone           -72.881070
fr_nitroso           -82.541114
fr_phos_acid        -117.838316
fr_azo              -136.178933
fr_diazo            -235.564039
Length: 201, dtype: float64

In [109]:
filtered_features = rfe_pipe.get_feature_names_out()
X_rfe = X[filtered_features]
X_rfe.info()

<class 'pandas.DataFrame'>
RangeIndex: 8763 entries, 0 to 8762
Columns: 201 entries, energy to fr_urea
dtypes: float64(101), int64(100)
memory usage: 13.4 MB


In these 227 remaining features, I am curious how many QM features are still left. 

In [110]:
with open("../data/properties.json", "r") as f:
    data = json.load(f)

qm_feature_names = [feature["name_of_property"] for feature in data]

In [111]:
qm_remaining_features = X_rfe.columns[X_rfe.columns.isin(qm_feature_names)]
print(len(qm_remaining_features))
qm_remaining_features

12


Index(['energy', 'atomization_energy', 'homo_lumo_gap', 'ionization_energy',
       'chemical_potential', 'molecular_dipole_norm',
       'molecular_polarizability_mean', 'enthalpy', 'zero_point_energy',
       'radius_of_gyration', 'molecular_volume', 'sterimol_L'],
      dtype='str')

This simple test of RFE shows that the best feature list is dominated by features that are 100% correlated with each other.

In [112]:
importance[qm_remaining_features]

energy                          -1.375216
atomization_energy              -0.404052
homo_lumo_gap                   -0.067946
ionization_energy               -0.145507
chemical_potential               0.333129
molecular_dipole_norm            0.148745
molecular_polarizability_mean   -0.144503
enthalpy                        -0.297502
zero_point_energy                1.817602
radius_of_gyration              -0.471013
molecular_volume                 0.261264
sterimol_L                      -0.337488
dtype: float64

In [113]:
X.corr()["solvation_energy_water"].sort_values(ascending=False)

solvation_energy_water          1.000000
solvation_energy_dmso           1.000000
solvation_energy_thf            0.999984
solvation_energy_cyclohexane    0.999864
ionization_energy               0.761719
                                  ...   
avg_partial_charge_water       -0.696032
avg_partial_charge             -0.696032
chemical_potential             -0.716627
SMR_VSA8                             NaN
SlogP_VSA9                           NaN
Name: solvation_energy_water, Length: 275, dtype: float64

In [114]:
X.corr()["molecular_sasa"].sort_values(ascending=False)

molecular_sasa                   1.000000
molecular_volume                 0.993700
entropy_300K                     0.990800
heat_capacity_300K               0.987794
Kappa1                           0.985153
                                   ...   
avg_overlap_integral            -0.696605
molecular_polarizability_mean   -0.719917
energy                          -0.969828
SMR_VSA8                              NaN
SlogP_VSA9                            NaN
Name: molecular_sasa, Length: 275, dtype: float64

In [115]:
part_charge_corr = X.corr()["avg_partial_charge"].abs()
part_charge_corr[part_charge_corr > 0.95].sort_values(ascending=False)

avg_partial_charge_cyclohexane    1.0
avg_partial_charge_dmso           1.0
avg_partial_charge_thf            1.0
avg_partial_charge                1.0
avg_partial_charge_water          1.0
Name: avg_partial_charge, dtype: float64

In [116]:
fukui_corr = X.corr()["avg_atomic_fukui_plus"].abs()
fukui_corr[fukui_corr > 0.95].sort_values(ascending=False)

avg_atomic_fukui_minus    1.000000
avg_atomic_fukui_plus     1.000000
avg_overlap_integral      0.955593
Name: avg_atomic_fukui_plus, dtype: float64